[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/information_theory/05_mutual_information/first_principles.ipynb)

# Topic 05: Mutual Information

## 1. First-Principles Intuition & Motivation

Topic 02 measured uncertainty *within* a variable and Topic 04 measured discrepancy *between* distributions.
Mutual information asks the question that sits between them: **how much does observing $Y$ tell us about $X$?**

Start with an operational thought experiment.
Before seeing $Y$, describing $X$ costs $H(X)$ bits on average.
After seeing $Y$, the residual description cost is $H(X \mid Y)$.
The savings,

$$
I(X; Y) = H(X) - H(X \mid Y)
$$

is by construction the number of bits per observation that $Y$ pays for.
Symmetry is not obvious from this reading — but it is true, and the proof is a two-line rearrangement.

### The Divergence Reading

There is an equivalent and arguably deeper definition: mutual information is the KL divergence between what actually happens and what would happen if the variables were independent,

$$
I(X; Y) = D_{\mathrm{KL}}\left(P_{XY} \parallel P_X P_Y\right)
$$

This form inherits everything Topic 04 established: nonnegativity by Jensen, equality iff $P_{XY} = P_X P_Y$ (independence), and the data-processing inequality.
It also survives the passage to continuous variables, where individual entropies become coordinate-dependent but the divergence does not.

### Why Not Correlation?

Correlation answers "how well does a straight line fit?"; mutual information answers "how much is there to know at all?".

| Property | Pearson correlation $\rho$ | Mutual information $I$ |
|---|---|---|
| Detects nonlinear dependence | no | yes |
| Zero implies independence | no | yes |
| Invariant under invertible reparameterization | no | yes |
| Range | $[-1, 1]$, signed | $[0, \infty)$, unsigned |
| Estimable from few samples in high dimension | yes | no |

The last row is the honest trade-off: mutual information is the right quantity and the hard one to measure.
Section 4 takes that difficulty seriously.

### The Venn Picture (and Its Limits)

For two variables the familiar diagram is exact: two circles of areas $H(X)$ and $H(Y)$ overlapping in $I(X; Y)$, with the union $H(X, Y)$ and the crescents $H(X \mid Y)$, $H(Y \mid X)$.
Every two-variable identity in this notebook is a statement about that picture.

The picture *fails* for three or more variables: the "triple overlap" $I(X; Y; Z) = I(X; Y) - I(X; Y \mid Z)$ can be negative (the XOR example), so it is not an area.
This is why conditional mutual information, not a three-circle diagram, is the correct tool for multivariate reasoning.

## 2. Rigorous Mathematical Definitions & Theorem Statements

### Definition 2.1 (Mutual Information)

For jointly distributed discrete variables $X, Y$,

$$
I(X; Y) = \sum_{x \in \mathcal{X}}\sum_{y \in \mathcal{Y}} p(x, y)\log\frac{p(x, y)}{p(x)p(y)} = \mathbb{E}_{p(x,y)}\left[\log\frac{p(x, y)}{p(x)p(y)}\right]
$$

The quantity inside the expectation, $i(x; y) = \log\frac{p(x, y)}{p(x)p(y)}$, is the **pointwise mutual information** (PMI); unlike its average it may be negative.
For continuous variables replace sums by integrals over densities; the definition is unchanged.

### Definition 2.2 (Conditional Mutual Information)

$$
I(X; Y \mid Z) = \sum_{z} p(z)\, I(X; Y \mid Z = z) = \mathbb{E}_{p(x,y,z)}\left[\log\frac{p(x, y \mid z)}{p(x \mid z)\,p(y \mid z)}\right]
$$

Equivalently $I(X; Y \mid Z) = H(X \mid Z) - H(X \mid Y, Z)$: the information $Y$ carries about $X$ *beyond* what $Z$ already told us.

### Definition 2.3 (Channel Capacity)

For a channel with transition law $p(y \mid x)$, the capacity is the largest mutual information achievable by choosing the input distribution:

$$
C = \max_{p(x)} I(X; Y)
$$

measured in bits per channel use. Shannon's noisy-channel coding theorem states that reliable communication is possible at every rate below $C$ and impossible above it.

### Definition 2.4 (Markov Chain and Sufficiency)

Variables form a Markov chain $X \to Y \to Z$ if $p(z \mid x, y) = p(z \mid y)$, equivalently $I(X; Z \mid Y) = 0$.
A statistic $T(Y)$ is **sufficient** for $X$ if $X \to T(Y) \to Y$ is also a Markov chain, which is exactly the condition $I(X; Y) = I(X; T(Y))$.

### Theorem Statements

- **Theorem A (Equivalent forms)**: $I(X; Y) = H(X) - H(X \mid Y) = H(Y) - H(Y \mid X) = H(X) + H(Y) - H(X, Y) = D_{\mathrm{KL}}(P_{XY} \parallel P_X P_Y)$. In particular $I$ is symmetric and $I(X; X) = H(X)$.
- **Theorem B (Nonnegativity)**: $I(X; Y) \ge 0$ with equality iff $X$ and $Y$ are independent; likewise $I(X; Y \mid Z) \ge 0$.
- **Theorem C (Chain rule)**: $I(X_1, \dots, X_n; Y) = \sum_{i=1}^{n} I(X_i; Y \mid X_1, \dots, X_{i-1})$.
- **Theorem D (Data-processing inequality)**: if $X \to Y \to Z$ is a Markov chain then $I(X; Y) \ge I(X; Z)$, with equality iff $X \to Z \to Y$ also holds (i.e., $Z$ is sufficient).
- **Theorem E (Gaussian MI)**: for jointly Gaussian $(X, Y)$ with correlation $\rho$, $I(X; Y) = -\tfrac{1}{2}\ln(1 - \rho^2)$; the additive white Gaussian noise channel with power constraint $P$ and noise power $N$ has capacity $C = \tfrac{1}{2}\log_2\left(1 + P/N\right)$.
- **Theorem F (Fano's inequality)**: for any estimator $\hat{X} = g(Y)$ of $X$ over an alphabet of size $M$ with error probability $P_e$, $H(X \mid Y) \le H_b(P_e) + P_e \log(M - 1)$, hence $P_e \ge \frac{H(X) - I(X; Y) - \log 2}{\log M}$.
- **Theorem G (InfoNCE bound)**: for $K$ paired samples and any critic $f$, $I(X; Y) \ge \log K - \mathcal{L}_{\mathrm{NCE}}(f)$, so no such bound can certify more than $\log K$ nats.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1 (All the equivalent forms of $I(X; Y)$)

**Step 1 (split the log).** Start from the definition and use $p(x, y) = p(x \mid y)p(y)$ inside the logarithm:

$$
I(X; Y) = \sum_{x, y} p(x, y)\log\frac{p(x \mid y)}{p(x)} = \sum_{x, y} p(x, y)\log\frac{1}{p(x)} - \sum_{x, y} p(x, y)\log\frac{1}{p(x \mid y)}
$$

**Step 2 (recognize the entropies).** The first sum marginalizes $y$ away and gives $H(X)$; the second is by definition $H(X \mid Y)$:

$$
I(X; Y) = H(X) - H(X \mid Y)
$$

**Step 3 (symmetry).** The joint form $\log\frac{p(x,y)}{p(x)p(y)}$ is symmetric in its arguments, so repeating Steps 1–2 with the roles of $X$ and $Y$ exchanged gives $I(X; Y) = H(Y) - H(Y \mid X)$.

**Step 4 (the union form).** Substituting the chain rule $H(X \mid Y) = H(X, Y) - H(Y)$ into Step 2:

$$
I(X; Y) = H(X) + H(Y) - H(X, Y)
$$

**Step 5 (the divergence form).** The definition is literally the KL divergence between the joint $P_{XY}$ and the product $P_X P_Y$ evaluated on the same alphabet.

$$
\boxed{I(X; Y) = H(X) - H(X \mid Y) = H(Y) - H(Y \mid X) = H(X) + H(Y) - H(X, Y) = D_{\mathrm{KL}}(P_{XY} \parallel P_X P_Y)}
$$

Setting $Y = X$ gives $H(X \mid X) = 0$ and hence $I(X; X) = H(X)$: entropy is self-information in the literal sense. $\blacksquare$

### Proof 3.2 (Nonnegativity, and $I = 0$ iff independence)

**Claim.** $I(X; Y) \ge 0$, with equality iff $p(x, y) = p(x)p(y)$ for all $(x, y)$.

**Step 1 (negate and apply Jensen).** Write

$$
-I(X; Y) = \sum_{x, y} p(x, y)\log\frac{p(x)p(y)}{p(x, y)} = \mathbb{E}_{p(x,y)}\left[\log R\right], \qquad R = \frac{p(X)p(Y)}{p(X, Y)}
$$

The logarithm is strictly concave, so Jensen's inequality gives $\mathbb{E}\left[\log R\right] \le \log \mathbb{E}\left[R\right]$.

**Step 2 (the mean of the ratio).**

$$
\mathbb{E}\left[R\right] = \sum_{x, y} p(x, y)\frac{p(x)p(y)}{p(x, y)} = \sum_{x, y} p(x)p(y) = 1
$$

(the sum runs over the support of $p(x,y)$, so it is at most 1, with equality when the product measure charges no extra points).

**Step 3 (conclude).** $-I(X; Y) \le \log 1 = 0$, hence $I(X; Y) \ge 0$.

**Step 4 (equality).** Strict concavity forces $R$ to be almost surely constant, and Step 2 pins the constant to 1: $p(x, y) = p(x)p(y)$ everywhere, i.e., independence.

**Corollary (conditioning reduces entropy).** Combining with Proof 3.1, $H(X \mid Y) \le H(X)$ — *on average*, observing $Y$ never increases uncertainty about $X$, though a particular value $Y = y$ certainly can.

$$
\boxed{I(X; Y) \ge 0, \quad \text{equality} \iff X \perp Y; \qquad H(X \mid Y) \le H(X)}
$$

$\blacksquare$

### Proof 3.3 (Chain rule for mutual information)

**Claim.** $I(X_1, X_2; Y) = I(X_1; Y) + I(X_2; Y \mid X_1)$, and by induction $I(X_1, \dots, X_n; Y) = \sum_i I(X_i; Y \mid X_{\lt i})$.

**Step 1 (start from the entropy form).**

$$
I(X_1, X_2; Y) = H(X_1, X_2) - H(X_1, X_2 \mid Y)
$$

**Step 2 (apply the entropy chain rule to both terms).**

$$
H(X_1, X_2) = H(X_1) + H(X_2 \mid X_1), \qquad H(X_1, X_2 \mid Y) = H(X_1 \mid Y) + H(X_2 \mid X_1, Y)
$$

**Step 3 (regroup).**

$$
I(X_1, X_2; Y) = \underbrace{\left[H(X_1) - H(X_1 \mid Y)\right]}_{I(X_1; Y)} + \underbrace{\left[H(X_2 \mid X_1) - H(X_2 \mid X_1, Y)\right]}_{I(X_2; Y \mid X_1)}
$$

**Step 4 (induction).** Treating $(X_1, \dots, X_{n-1})$ as a single variable and repeating gives the general statement.

$$
\boxed{I(X_1, \dots, X_n; Y) = \sum_{i=1}^{n} I\left(X_i; Y \mid X_1, \dots, X_{i-1}\right)}
$$

**Interpretation.** Features contribute information *sequentially and contextually*: the value of $X_2$ is measured only after $X_1$ has been accounted for, which is exactly why greedy forward selection (and the "information gain" of a decision-tree split) is a chain-rule computation. $\blacksquare$

### Proof 3.4 (Data-processing inequality)

**Claim.** If $X \to Y \to Z$ is a Markov chain, then $I(X; Y) \ge I(X; Z)$.

**Step 1 (expand $I(X; Y, Z)$ two ways using the chain rule).**

$$
I(X; Y, Z) = I(X; Y) + I(X; Z \mid Y) = I(X; Z) + I(X; Y \mid Z)
$$

**Step 2 (use the Markov property).** $X \to Y \to Z$ means $X$ and $Z$ are conditionally independent given $Y$, i.e., $I(X; Z \mid Y) = 0$. The first expansion collapses to

$$
I(X; Y, Z) = I(X; Y)
$$

**Step 3 (nonnegativity of the leftover).** By Proof 3.2 applied conditionally, $I(X; Y \mid Z) \ge 0$. Substituting into the second expansion:

$$
I(X; Y) = I(X; Z) + I(X; Y \mid Z) \ge I(X; Z)
$$

**Step 4 (equality condition).** Equality holds iff $I(X; Y \mid Z) = 0$, i.e., $X \to Z \to Y$ is also a Markov chain — precisely the statement that $Z$ is a *sufficient statistic* of $Y$ for $X$.

$$
\boxed{X \to Y \to Z \implies I(X; Y) \ge I(X; Z), \text{ equality iff } Z \text{ is sufficient}}
$$

**Corollaries.** $I(X; g(Y)) \le I(X; Y)$ for any (possibly random) function $g$; in a feedforward network $X \to Z_1 \to Z_2 \to \cdots \to \hat{Y}$, the information about the input is nonincreasing with depth. $\blacksquare$

### Proof 3.5 (Gaussian mutual information and the AWGN capacity)

**Part 1 (bivariate Gaussian).** Let $(X, Y)$ be jointly Gaussian with unit variances and correlation $\rho$. Differential entropies of Gaussians are

$$
h(X) = \tfrac{1}{2}\ln(2\pi e), \qquad h(X, Y) = \tfrac{1}{2}\ln\left((2\pi e)^2 \det \Sigma\right), \quad \det\Sigma = 1 - \rho^2
$$

Using the union form of Proof 3.1 (valid for differential entropies because the unit-dependent terms cancel):

$$
I(X; Y) = h(X) + h(Y) - h(X, Y) = -\tfrac{1}{2}\ln\left(1 - \rho^2\right)
$$

which is 0 at $\rho = 0$ and diverges as $\vert \rho \vert \to 1$ — perfect dependence carries infinite information in the continuous world.

**Part 2 (AWGN capacity).** Let $Y = X + N$ with $N \sim \mathcal{N}(0, N_0)$ independent of $X$, subject to $\mathbb{E}[X^2] \le P$. Then

$$
I(X; Y) = h(Y) - h(Y \mid X) = h(Y) - h(N)
$$

since given $X$, $Y$ is just $N$ shifted. Now $\mathrm{Var}(Y) \le P + N_0$, and among all distributions with a given variance the Gaussian maximizes differential entropy, so

$$
h(Y) \le \tfrac{1}{2}\ln\left(2\pi e (P + N_0)\right)
$$

with equality attainable by taking $X \sim \mathcal{N}(0, P)$, which makes $Y$ Gaussian. Therefore

$$
C = \max_{p(x)} I(X; Y) = \tfrac{1}{2}\ln\frac{P + N_0}{N_0} = \tfrac{1}{2}\log_2\left(1 + \frac{P}{N_0}\right) \text{ bits per use}
$$

$$
\boxed{I_{\text{Gauss}} = -\tfrac{1}{2}\ln(1 - \rho^2), \qquad C_{\mathrm{AWGN}} = \tfrac{1}{2}\log_2\left(1 + \mathrm{SNR}\right)}
$$

$\blacksquare$

### Proof 3.6 (Fano's inequality)

**Claim.** Let $X$ take values in an alphabet of size $M$, let $\hat{X} = g(Y)$, and let $P_e = \Pr\left[\hat{X} \neq X\right]$. Then

$$
H(X \mid Y) \le H_b(P_e) + P_e \log(M - 1)
$$

**Step 1 (introduce the error indicator).** Let $E = \mathbf{1}\{\hat{X} \neq X\}$. Expand $H(E, X \mid \hat{X})$ with the chain rule in two orders:

$$
H(E, X \mid \hat{X}) = H(E \mid \hat{X}) + H(X \mid E, \hat{X}) = H(X \mid \hat{X}) + \underbrace{H(E \mid X, \hat{X})}_{= 0}
$$

The last term vanishes because $E$ is a deterministic function of $(X, \hat{X})$.

**Step 2 (bound each surviving term).** $H(E \mid \hat{X}) \le H(E) = H_b(P_e)$ since conditioning reduces entropy. For the other,

$$
H(X \mid E, \hat{X}) = \Pr[E{=}0]\,H(X \mid \hat{X}, E{=}0) + \Pr[E{=}1]\,H(X \mid \hat{X}, E{=}1) \le 0 + P_e\log(M - 1)
$$

because on the no-error event $X = \hat{X}$ is determined, and on the error event $X$ is confined to the $M - 1$ values different from $\hat{X}$.

**Step 3 (combine and apply the DPI).**

$$
H(X \mid \hat{X}) \le H_b(P_e) + P_e\log(M - 1)
$$

and since $X \to Y \to \hat{X}$ is a Markov chain, $H(X \mid Y) \le H(X \mid \hat{X})$.

**Step 4 (solve for $P_e$).** Using $H(X \mid Y) = H(X) - I(X; Y)$ and $H_b \le \log 2$:

$$
P_e \ge \frac{H(X) - I(X; Y) - \log 2}{\log M}
$$

$$
\boxed{P_e \ge \frac{H(X) - I(X; Y) - \log 2}{\log M}}
$$

**Interpretation.** Accuracy is capped by information: a classifier whose features carry only $I(X; Y)$ bits about a label with $H(X)$ bits of entropy *cannot* be accurate, regardless of architecture or training budget. Fano is the standard tool for minimax lower bounds in statistics. $\blacksquare$

### Proof 3.7 (The InfoNCE lower bound and its $\log K$ ceiling)

**Setup.** Draw one positive pair $(x_1, y_1) \sim p(x, y)$ and $K - 1$ negatives $y_2, \dots, y_K \sim p(y)$ independently. A critic $f(x, y)$ scores pairs, and the contrastive loss is

$$
\mathcal{L}_{\mathrm{NCE}} = -\mathbb{E}\left[\log\frac{e^{f(x_1, y_1)}}{\frac{1}{K}\sum_{j=1}^{K} e^{f(x_1, y_j)}}\right]
$$

**Step 1 (optimal critic).** The loss is a categorical cross-entropy for identifying which of the $K$ candidates is the true partner. The Bayes-optimal posterior over "index $j$ is the positive" is

$$
\Pr\left[j \text{ positive} \mid x_1, y_{1:K}\right] = \frac{p(y_j \mid x_1)/p(y_j)}{\sum_{k} p(y_k \mid x_1)/p(y_k)}
$$

so the optimal critic is the log density ratio $f^{*}(x, y) = \log\frac{p(y \mid x)}{p(y)} + c(x)$ — exactly the pointwise mutual information up to a per-$x$ constant.

**Step 2 (substitute the optimum).** Plugging $f^{*}$ in and writing $r(x,y) = \frac{p(y \mid x)}{p(y)}$:

$$
\mathcal{L}_{\mathrm{NCE}}(f^{*}) = \mathbb{E}\left[\log\left(\frac{1}{K}\left(r(x_1, y_1) + \sum_{j=2}^{K} r(x_1, y_j)\right)\right)\right] - \mathbb{E}\left[\log r(x_1, y_1)\right]
$$

The second expectation is exactly $I(X; Y)$.

**Step 3 (bound the first term).** Since $\mathbb{E}_{y_j \sim p(y)}\left[r(x_1, y_j)\right] = 1$ for the negatives, Jensen's inequality gives

$$
\mathbb{E}\left[\log\left(\tfrac{1}{K}\left(r_1 + \textstyle\sum_{j \ge 2} r_j\right)\right)\right] \le \log\left(\tfrac{1}{K}\left(\mathbb{E}[r_1] + (K-1)\right)\right)
$$

and dropping the positive term's contribution in the crudest way (replacing the inner sum by its $(K-1)$ mean and keeping $r_1 \ge 0$) yields the standard conclusion

$$
I(X; Y) \ge \log K - \mathcal{L}_{\mathrm{NCE}}(f)
$$

for every critic $f$, with the gap closing as $f \to f^{*}$.

**Step 4 (the ceiling).** Because $\mathcal{L}_{\mathrm{NCE}} \ge 0$ (it is a cross-entropy over $K$ classes and cannot be negative in expectation), the bound can never certify more than $\log K$ nats:

$$
\boxed{I(X; Y) \ge \log K - \mathcal{L}_{\mathrm{NCE}}, \qquad \text{bound} \le \log K}
$$

**Interpretation.** With a batch of $K = 256$, InfoNCE can prove at most $\log 256 = 5.55$ nats $= 8$ bits of mutual information — even if the true value is hundreds of bits. This is why contrastive objectives are excellent *training signals* and poor *measurements*, and why practitioners chase ever-larger batches and memory banks. $\blacksquare$

## 4. Computational & Algorithmic Insights

### Discrete Estimation: The Bias You Will Definitely Hit

The plug-in ("maximum likelihood") estimator computes empirical frequencies over a contingency table and evaluates the formula. It is *systematically biased upward*:

$$
\mathbb{E}\left[\hat{I}_{\text{plug-in}}\right] \approx I(X; Y) + \frac{(\vert \mathcal{X} \vert - 1)(\vert \mathcal{Y} \vert - 1)}{2N} \text{ nats}
$$

Consequences and remedies:

- **Independent variables never estimate to zero.** With $10 \times 10$ bins and $N = 200$ samples the expected spurious MI is about $0.20$ nats — larger than many real effects.
- **Miller–Madow correction**: subtract the analytic bias term above; cheap and usually sufficient for modest alphabets.
- **Permutation null**: shuffle $Y$ against $X$ many times, recompute $\hat{I}$, and report the observed value against that null distribution. This is the only defensible way to claim "the dependence is real".
- **Bin count matters more than bin placement**: doubling the resolution doubles the bias while adding little signal; use equal-mass (quantile) bins and keep $\vert \mathcal{X} \vert\vert \mathcal{Y} \vert \ll N$.

### Continuous Estimation: KSG and Its Limits

The Kraskov–Stögbauer–Grassberger (KSG) estimator avoids explicit density estimation by working with $k$-nearest-neighbor distances in the joint space and counting how many points fall within the corresponding marginal radii:

$$
\hat{I}_{\mathrm{KSG}} = \psi(k) + \psi(N) - \frac{1}{N}\sum_{i=1}^{N}\left[\psi(n_x(i) + 1) + \psi(n_y(i) + 1)\right]
$$

where $\psi$ is the digamma function and $n_x(i), n_y(i)$ are the marginal neighbor counts for point $i$.

Practical notes:

- **Choice of $k$** trades bias (small $k$) against variance (large $k$); $k \in \{3, \dots, 10\}$ is standard.
- **Scale sensitivity**: distances mix the coordinates, so standardize or whiten first; KSG is *not* invariant to coordinatewise rescaling in practice even though $I$ is in theory.
- **Dimension**: reliability degrades badly beyond roughly 5–10 dimensions — the curse of dimensionality applies to the neighbor statistics.
- **Ties and discreteness**: add tiny jitter to break ties, or use a discrete/continuous hybrid estimator.

### Variational Estimators: MINE, NWJ, InfoNCE

When the variables are high-dimensional, replace estimation by optimization; each bound trains a critic network $f_\theta(x, y)$.

| Bound | Objective | Bias/variance | Ceiling |
|---|---|---|---|
| Donsker–Varadhan (MINE) | $\mathbb{E}_{p(x,y)}[f] - \log\mathbb{E}_{p(x)p(y)}\left[e^{f}\right]$ | low bias, high variance; log-of-mean-exp needs EMA debiasing | none in theory |
| NWJ / f-divergence | $\mathbb{E}_{p(x,y)}[f] - e^{-1}\mathbb{E}_{p(x)p(y)}\left[e^{f}\right]$ | unbiased objective, still high variance | none in theory |
| InfoNCE / CPC | $\log K - \mathcal{L}_{\mathrm{NCE}}$ | low variance, strongly biased low | $\log K$ |
| Barber–Agakov | $H(X) - \mathbb{E}\left[-\log q_\phi(x \mid y)\right]$ | needs a tractable decoder and known $H(X)$ | none in theory |

Rules of thumb: use InfoNCE when you want a *stable training signal*; use DV/NWJ with large batches and EMA smoothing when you want an *estimate*; and never report a variational MI number without also reporting $K$, the batch size, and a shuffled-pairs control.

### Feature Selection with Mutual Information

- **Filter ranking**: score each candidate feature by $I(X_j; Y)$ and keep the top $m$. Fast, but blind to redundancy — two copies of the same excellent feature both score highly.
- **mRMR (minimum redundancy, maximum relevance)**: greedily add the feature maximizing $I(X_j; Y) - \frac{1}{\vert S \vert}\sum_{k \in S} I(X_j; X_k)$, penalizing overlap with already-selected features.
- **JMI / CMIM**: use conditional mutual information $I(X_j; Y \mid X_k)$ directly, which is the chain rule of Proof 3.3 applied greedily and is theoretically the right criterion.
- **Decision trees**: the information gain of a split on feature $A$ is exactly $I(Y; A) = H(Y) - H(Y \mid A)$, and the gain ratio divides it by $H(A)$ to counteract the bias toward high-cardinality features.

## 5. Real-World Physics & AI/ML Applications

### Representation Learning and Self-Supervision

Contrastive methods (CPC, SimCLR, CLIP) train encoders by maximizing an InfoNCE lower bound on the mutual information between two views of the same datum — two augmentations of an image, an image and its caption, or the present and the future of a sequence.
Proof 3.7 explains three empirical regularities at once: large batches help (the $\log K$ ceiling), the learned critic behaves like a log density ratio (so its logits are calibrated similarity scores), and the achieved MI estimate is a floor rather than a measurement.

The **InfoMax principle** — choose the representation maximizing $I(X; Z)$ subject to a capacity constraint — predates deep learning (Linsker, 1988) and reappears in Deep InfoMax, InfoGAN's latent-code regularizer, and the information bottleneck of Topic 06.

### Communication, Neuroscience, and Physics

- **Channel coding**: capacity $C = \max_p I(X; Y)$ is the sharp boundary between achievable and impossible rates; modern LDPC and polar codes approach it within fractions of a dB.
- **Water-filling**: for parallel Gaussian channels, maximizing total MI under a power budget yields the water-filling solution — the same Lagrangian structure as rate allocation in learned compression.
- **Neural coding**: mutual information between a stimulus and a spike train quantifies how much a neuron actually transmits; efficient-coding theories posit that sensory systems maximize $I$ subject to metabolic constraints.
- **Statistical physics**: mutual information between subsystems measures correlations; in spin systems it detects phase transitions, and the mutual information of a bipartition is the classical analogue of entanglement entropy.

### Diagnostics, Fairness, and Causality Caveats

- **Feature auditing**: $I(\text{sensitive attribute}; \hat{Y})$ quantifies leakage of a protected attribute into predictions; the DPI guarantees that a representation with $I(S; Z) = 0$ yields $I(S; \hat{Y}) = 0$ for *any* downstream head — a strong, architecture-independent fairness certificate.
- **Layer analysis**: measuring $I(X; Z_\ell)$ and $I(Y; Z_\ell)$ across layers is the empirical program behind information-plane studies; the DPI says the first is nonincreasing in $\ell$, so any reported increase is an estimation artifact (usually binning).
- **Not causality**: $I(X; Y) \gt 0$ is symmetric and says nothing about direction or confounding. Interventional or conditional-independence machinery is required before any causal claim; conditional MI $I(X; Y \mid Z)$ is a *test statistic* for conditional independence, not a proof of mechanism.

## 6. Canonical Literature Mapping & References

| Concept in this notebook | Canonical source |
|---|---|
| MI definition, equivalent forms, chain rule | Shannon (1948); Cover & Thomas (2006), Chapter 2 |
| Data-processing inequality, sufficiency | Cover & Thomas (2006), Section 2.8 |
| Channel capacity, AWGN, water-filling | Shannon (1948); Cover & Thomas (2006), Chapters 7, 9 |
| Fano's inequality and minimax lower bounds | Fano (1961); Cover & Thomas (2006), Section 2.10 |
| Plug-in bias, Miller–Madow correction | Miller (1955); Paninski (2003), *Neural Computation* |
| $k$-NN mutual information estimation | Kraskov, Stögbauer & Grassberger (2004), *Phys. Rev. E* |
| Variational MI lower bounds | Barber & Agakov (2003); Belghazi et al. (2018), MINE |
| InfoNCE and contrastive predictive coding | van den Oord, Li & Vinyals (2018) |
| Limits of MI estimation, the $\log K$ ceiling | Poole et al. (2019); McAllester & Stratos (2020) |
| MI-based feature selection (mRMR, JMI) | Peng, Long & Ding (2005); Brown et al. (2012), JMLR |
| InfoMax principle in neural systems | Linsker (1988); Bell & Sejnowski (1995) |